In [1]:
import igvf_and_terra_api_tools as api_tools
import portal_to_terra_input_from_anaset as portal2terra_transfer
import terra_to_portal_posting as terra2portal_transfer

import firecloud.api as fapi

import importlib
import pandas as pd
import subprocess

In [14]:
importlib.reload(portal2terra_transfer)

<module 'portal_to_terra_input_from_anaset' from '/Users/zheweishen/IGVF/IGVF_Repos/sc-pipeline-management/src/portal_to_terra_input_from_anaset.py'>

### Set up some params for pushing and posting

In [2]:
# Set params
terra_namespace = 'DACC_ANVIL'
terra_workspace = 'Playground for IGVF Single-Cell Data Processing'
test_run_terra_workspace = 'IGVF Single-Cell Data Processing'

igvf_sandbox_analysis_sets = ['TSTDS33660419', 'TSTDS12024147']
igvf_production_analysis_sets = ['IGVFDS5316CVFR', 'IGVFDS4768JVVU', 'IGVFDS8343PSUS']
pipeline_test_run1= ['IGVFDS5316CVFR']
pipeline_test_run2= ['IGVFDS8343PSUS']

In [4]:
# Set up IGVF query API (Sandbox or Production)
igvf_api_sandbox = api_tools.get_igvf_auth_and_api(igvf_site='sandbox')
igvf_api_production = api_tools.get_igvf_auth_and_api(igvf_site='production')

# IGVF util
iu_conn_sandbox = api_tools.get_igvf_utils_connection(igvf_utils_mode='sandbox', submission_mode=True)

2025-02-26 14:33:56,984:iu_debug:	submission=True: In submission mode.


In [5]:
# If FireCloud session somehow expires
fapi._set_session()

### Generate the Terra data table from IGVF portal

#### Get the input parameter full table

In [15]:
# Construct the input table from portal based on analysis set accessions
portal_to_terra_input_table = portal2terra_transfer.generate_pipeline_input_table(query_analysis_set_accs=igvf_production_analysis_sets, igvf_api=igvf_api_production)
portal_to_terra_input_table

In [16]:
portal_to_terra_input_table

,analysis_set_acc,atac_MeaSetIDs,rna_MeaSetIDs,subpool_id,taxa,genome_assembly,genome_ref,atac_read1_accessions,atac_read2_accessions,atac_barcode_accessions,...,atac_read2,atac_barcode,rna_read1,rna_read2,atac_barcode_inclusion_list,atac_read_format,rna_barcode_inclusion_list,rna_read_format,onlist_mapping,possible_errors
0,IGVFDS5316CVFR,"[IGVFDS9728TWXO, IGVFDS7747XDUI, IGVFDS8700PWZJ]","[IGVFDS9620KXMW, IGVFDS6721DTCZ, IGVFDS4161OUYL]",IGVFSM8883WYCA,Mus musculus,GRCm39,<mouse genome tsv>,"[IGVFFI9201WYHO, IGVFFI0137ERGS, IGVFFI7270DMR...","[IGVFFI9305UBWA, IGVFFI3119TNSE, IGVFFI7385RIW...","[IGVFFI9305UBWA, IGVFFI3119TNSE, IGVFFI7385RIW...",...,[https://api.data.igvf.org/sequence-files/IGVF...,[https://api.data.igvf.org/sequence-files/IGVF...,[https://api.data.igvf.org/sequence-files/IGVF...,[https://api.data.igvf.org/sequence-files/IGVF...,None,"bc:65:72,bc:103:110,bc:141:148,r1:0:49,r2:0:49",None,"1,65,73,1,103,111,1,141,149:1,0,10:0,0,50",False,[Error: Seqspec onlist files are different fro...
1,IGVFDS4768JVVU,[IGVFDS4021UKPE],[IGVFDS1449TKWZ],IGVFSM8062LNQW,Mus musculus,GRCm39,<mouse genome tsv>,[IGVFFI0360VNTD],[IGVFFI5699KRFZ],[IGVFFI2788FRNF],...,[https://api.data.igvf.org/sequence-files/IGVF...,[https://api.data.igvf.org/sequence-files/IGVF...,[https://api.data.igvf.org/sequence-files/IGVF...,[https://api.data.igvf.org/sequence-files/IGVF...,None,"bc:8:23:-,r1:0:49,r2:0:49",final_barcode_list/IGVFDS4768JVVU_rna_final_ba...,"0,0,16:0,16,28:2,0,90",True,[Error: Seqspec onlist files are different fro...
2,IGVFDS8343PSUS,[IGVFDS2289SZHO],[IGVFDS0602EDSF],IGVFSM1147GCSX,Mus musculus,GRCm39,<mouse genome tsv>,[IGVFFI7536FOCA],[IGVFFI0430XHJG],[IGVFFI3738RLGC],...,[https://api.data.igvf.org/sequence-files/IGVF...,[https://api.data.igvf.org/sequence-files/IGVF...,[https://api.data.igvf.org/sequence-files/IGVF...,[https://api.data.igvf.org/sequence-files/IGVF...,final_barcode_list/IGVFDS8343PSUS_atac_final_b...,"bc:8:23:-,r1:0:49,r2:0:49",None,"0,0,16:0,16,28:2,0,90",True,[Error: Seqspec onlist files are different fro...


#### Upload to Terra

In [66]:
# Upload this to Terra under the name 'DACC_Tester'
portal_to_terra_entity_type = 'Pipeline_run_tester'
api_tools.upload_portal_input_tsv_to_terra(terra_namespace=terra_namespace,
                                             terra_workspace=terra_workspace,
                                             terra_etype=portal_to_terra_entity_type,
                                             portal_input_table=portal_to_terra_input_table,
                                             verbose=True
                                            )

<Response at 12af7b450: <Response [200]>> 
  _content: <str at 0x12b03f5f0>: "b'Pipeline_run_tester'"
  _content_consumed: True
  _next: None
  status_code: 200
  headers: <CaseInsensitiveDict at 12b2a8e90: {'Date': 'Fri, 21 Feb 2025 05:30:35 GMT', 'Server': 'Apache', 'X-Frame-Options': 'SAMEORIGIN', 'X-XSS-Protection': '1; mode=block', 'X-Content-Type-Options': 'nosniff', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains', 'Referrer-Policy': 'strict-origin-when-cross-origin', 'Cache-control': 'no-store', 'Pragma': 'no-cache', 'Access-Control-Allow-Origin': '*', 'Access-Control-Allow-Headers': 'authorization,content-type,accept,origin,x-app-id', 'Access-Control-Allow-Methods': 'GET,POST,PUT,PATCH,DELETE,OPTIONS,HEAD', 'Access-Control-Max-Age': '1728000', 'Content-Type': 'text/plain; charset=UTF-8', 'Content-Length': '19', 'Via': '1.1 google', 'Alt-Svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000'}>
    object contents suppressed (instance from different module)
  r

In [19]:
terra_workspace

'Playground for IGVF Single-Cell Data Processing'

### POSTing pipeline output data to IGVF data portal

#### Set up POST params and retrieve data table from Terra

In [ ]:
# For posting (this mock table mixes the input table because it has IGVF accessions and some output from Team 2 because I need files to try posting)
terra_to_portal_etype = 'DACC_mockTeam_2_tester_merged'
Sandbox_test_analysis_set = 'TSTDS33660419'

In [ ]:
# Generate the input data table for Terra
terra_to_portal_post_datatable = api_tools.get_terra_tsv_data_table(terra_namespace=terra_namespace,
                                                                    terra_workspace=terra_workspace,
                                                                    terra_etype=terra_to_portal_etype
                                                                   )
terra_to_portal_post_datatable.head(2)

#### Post all successful run results to the portal

In [ ]:
# Run all posting jobs
terra_to_portal_post_runs = terra2portal_transfer.post_all_successful_runs(igvf_api=igvf_api_sandbox, igvf_utils_api=iu_conn_sandbox, upload_file=False, full_terra_data_table=terra_to_portal_post_datatable)

In [ ]:
# Get the full summary
terra_to_portal_post_summary = terra2portal_transfer.summarize_post_status(post_results=terra_to_portal_post_runs)

In [ ]:
# Update the original output table with brief post summary
terra_to_portal_post_summary = terra2portal_transfer.add_post_status_summary_to_output_data_table(full_terra_data_table=terra_to_portal_post_datatable, post_status_df=terra_to_portal_post_summary)

#### Optional

In [ ]:
# Upload the run results to Terra
terra_to_portal_postres_etype = 'DACC_mockTeam_2_tester_merged'
api_tools.upload_output_post_res_to_terra(terra_namespace=terra_namespace,
                                          terra_workspace=terra_workspace,
                                          terra_etype=terra_to_portal_postres_etype,
                                          verbose=True
                                         )